In [170]:
import os
import random

import cv2
import numpy as np
from pathlib import Path

In [194]:
def extract_patches(img_folder: str, lbl_folder: str, out_path: str, ball_id: int, patch_size: int, negative_samples: bool):
    if negative_samples:
        (Path(out_path)/"no-object").mkdir(parents=True, exist_ok=True)
    for lbl in os.listdir(lbl_folder):
        if not lbl.endswith(".txt"):
            continue
        img_path = os.path.join(img_folder, lbl.replace(".txt", ".png"))
        assert os.path.exists(img_path), f"Image file for label {lbl} not found."
        # print(f"img_path: {img_path}\nlbl_path: {os.path.join(lbl_folder, lbl)}")
        with open(os.path.join(lbl_folder, lbl)) as f:
            ball_lines = [l for l in f.readlines() if l.startswith(str(ball_id))]

            ball_patches =  [[float(i) for i in line.split(" ")[1:]] for line in ball_lines]
            if len(ball_patches) > 0:
                image = cv2.imread(img_path)

                for i, bp in enumerate(ball_patches):
                    # print(bp)
                    x = min(max(0., bp[0]), 1.)
                    y = min(max(0., bp[1]), 1.)
                    w = min(max(0., bp[2]), 1.)
                    h = min(max(0., bp[3]), 1.)
                    # w =1
                    # h=1
                    bp2 = [int(x * image.shape[1]), int(y * image.shape[0]), int(w*image.shape[1]), int(h*image.shape[0])]


                    margin = 0.1
                    xul = bp2[0]-bp2[2]//2-margin*bp2[2]
                    xlr = bp2[0]+bp2[2]//2+margin*bp2[2]
                    yul = bp2[1]-bp2[3]//2-margin*bp2[3]
                    ylr = bp2[1]+bp2[3]//2+margin*bp2[3]

                    # clamping patch in xyxy format to prevent out of bounds
                    xul = max(int(xul), 0)
                    yul = max(int(yul), 0)
                    xlr = min(int(xlr), image.shape[1])
                    ylr = min(int(ylr), image.shape[0])

                    # enforcing squareness by expanding shorter size.
                    # Deviate off-center if at a border
                    wh = max(xlr-xul, ylr-yul)
                    if xul == 0:
                        xlr = xul + wh
                    elif xlr == image.shape[1]:
                        xul = xlr - wh
                    # handle x and y separately to allow for bidirectionally off-center patches when at borders
                    if yul == 0:
                        ylr = yul + wh
                    elif ylr == image.shape[0]:
                        yul = ylr - wh
                    else:
                        # expand equally in both directions to keep center
                        xul -= (wh - (xlr-xul))//2
                        xlr += (wh - (xlr-xul))//2
                        yul -= (wh - (ylr-yul))//2
                        ylr += (wh - (ylr-yul))//2

                        # manually update lr after expansion to prevent rounding issues
                        xlr = xul + wh
                        ylr = yul + wh

                    patch_image = image[yul:ylr, xul:xlr]
                    patch_image = cv2.resize(patch_image, (patch_size, patch_size), interpolation=cv2.INTER_NEAREST)

                    if negative_samples:
                        dist_left = xul
                        dist_right = image.shape[1] - xlr
                        dist_top = yul
                        dist_bottom = image.shape[0] - ylr

                        if dist_left > wh and random.random() < 0.5:
                            no_ball_x_ul = random.randint(0, xul-wh)
                        elif dist_right > wh:
                            no_ball_x_ul = random.randint(xlr, image.shape[1]-wh)
                        else:
                            # repeating case 1 (to bypass randomness)
                            no_ball_x_ul = random.randint(0, xul-wh)

                        if dist_top > wh and random.random() < 0.5:
                            no_ball_y_ul = random.randint(0, yul-wh)
                        elif dist_bottom > wh:
                            no_ball_y_ul = random.randint(ylr, image.shape[0]-wh)
                        else:
                            # repeating case 1 (to bypass randomness)
                            no_ball_y_ul = random.randint(0, yul-wh)

                        no_obj_patch = image[no_ball_y_ul:no_ball_y_ul+wh, no_ball_x_ul:no_ball_x_ul+wh]

                        no_obj_patch = cv2.resize(no_obj_patch, (patch_size, patch_size), interpolation=cv2.INTER_NEAREST)
                        cv2.imwrite(os.path.join(out_path, "no-object", f"{lbl.removesuffix('.txt')}_{i}_no_obj.png"), no_obj_patch)
                    cv2.imwrite(os.path.join(out_path, f"{lbl.removesuffix('.txt')}_{i}.png"), patch_image)


In [197]:
dataset = "/home/simon/Desktop/yolo-training/Cvat_Log_11_cls"
out_path = os.path.join(dataset, "ball_patches")
Path(out_path).mkdir(parents=True, exist_ok=True)
splits = ["train", "val"]
for split in splits:
    img_folder = os.path.join(dataset, split, "images")
    lbl_folder = os.path.join(dataset, split, "labels")

    extract_patches(img_folder, lbl_folder, out_path, ball_id=0, patch_size=32, negative_samples=True)